# Intel Core Ultra NPU: GNN Benchmarking Suite

Research pipeline for evaluating Graph Neural Networks (GNNs) on Intel Core Ultra NPU.
Run cells sequentially. Each visualization cell produces PNG + SVG outputs.

## Notebook Structure
1. **Setup** - Environment initialization
2. **Phase 1** - Model generation
3. **Phase 2** - Benchmarking (3 datasets, 100+ iterations)
4. **Phase 3** - Data analysis (4 analysis modules)
5. **Phase 4** - Visualizations (6 figures)
6. **Summary** - Output inventory

In [ ]:
import os
import sys

# ═══════════════════════════════════════════════════════════════
# CRITICAL: Disable ALL progress bars BEFORE any library imports
# Must run before tqdm/ogb/transformers are ever imported.
# ═══════════════════════════════════════════════════════════════
os.environ['TQDM_DISABLE'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Also silence Python logging from ogb/transformers
import logging
for _noisy in ('ogb', 'transformers', 'torch.onnx', 'onnx', 'nncf'):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)

# Auto-decline OGB dataset update prompts
import builtins
_orig_input = builtins.input
def _auto_decline_dataset_update(prompt: str):
    if "update the dataset now" in prompt.lower():
        print(f"{prompt}n")
        return "n"
    return _orig_input(prompt)
builtins.input = _auto_decline_dataset_update

# ═══════════════════════════════════════════════════════════════
# Normal setup begins below
# ═══════════════════════════════════════════════════════════════
import ctypes
import warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Suppress warnings
warnings.filterwarnings('ignore')

# Check admin rights
try:
    is_admin = bool(ctypes.windll.shell32.IsUserAnAdmin())
    print(f"Admin: {'Yes' if is_admin else 'No'}")
except:
    is_admin = False
    print("Admin check failed, assuming non-admin")

# Setup paths
RESULTS_DIR = Path("results")
MODELS_DIR = Path("models")
FIGURES_DIR = Path("results/figures")
DATA_DIR = Path("data")

for d in [RESULTS_DIR, MODELS_DIR, FIGURES_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Check model files
model_files = list(MODELS_DIR.glob("*_fp32.onnx"))
print(f"Found {len(model_files)} FP32 models: {[m.stem for m in model_files[:5]]}")
if len(model_files) > 5:
    print(f"... and {len(model_files)-5} more")

plt.style.use('default')
sns.set_theme(style="whitegrid")

# ═══════════════════════════════════════════════════════════════
# TEST MODE
# Set TEST_MODE = True  → fast smoke-test (low iterations, 1 repeat, 1 device, 1 dataset)
# Set TEST_MODE = False → full benchmarking (100+ iterations, 3 repeats, all devices & datasets)
# ═══════════════════════════════════════════════════════════════
TEST_MODE = False

# Safety switches to avoid driver/kernel crashes on some devices/models.
SKIP_GPU_INT8 = True
SKIP_NPU_INT8 = False
UNSTABLE_NPU_MODELS = {"GATv2", "GraphTransformer"}
SKIP_UNSTABLE_NPU_MODELS = True

# Per-model analysis controls
RUN_HW_COMPARISON = True
RUN_ENERGY_ANALYSIS = True

print(f"\n{'='*60}")
print(f"TEST_MODE = {TEST_MODE}")
print(f"SKIP_GPU_INT8 = {SKIP_GPU_INT8} | SKIP_NPU_INT8 = {SKIP_NPU_INT8}")
if SKIP_UNSTABLE_NPU_MODELS:
    print(f"SKIP_UNSTABLE_NPU_MODELS = True -> {sorted(UNSTABLE_NPU_MODELS)}")
else:
    print("SKIP_UNSTABLE_NPU_MODELS = False")
print(f"RUN_HW_COMPARISON = {RUN_HW_COMPARISON} | RUN_ENERGY_ANALYSIS = {RUN_ENERGY_ANALYSIS}")
if TEST_MODE:
    print("  Running in TEST mode (fast smoke-test)")
    print("  → Low iterations, 1 repeat, reduced device/dataset coverage")
    print("  → Set TEST_MODE = False for full production run")
else:
    print("  Running in FULL mode (production benchmarking)")
print(f"{'='*60}")

print("\nEnvironment ready.")

Admin: Yes
Found 14 FP32 models: ['APPNP_fp32', 'bert-tiny_fp32', 'efficientnet-b0_fp32', 'GATv2_fp32', 'GAT_fp32']
... and 9 more

TEST_MODE = False
  Running in FULL mode (production benchmarking)

Environment ready.


## Phase 1: Model Generation

Generate GNN and baseline models (FP32 + INT8 for all):

**GNN Models (9):**
- GCN (Graph Convolutional Network)
- GAT (Graph Attention Network)
- GATv2 (Graph Attention v2)
- GraphSAGE (Inductive representation learning)
- GIN (Graph Isomorphism Network)
- APPNP (Approximate Personalized Propagation)
- GraphTransformer (Transformer-based GNN)
- MPNN (Message Passing Neural Network)
- SGC (Simplified Graph Convolution)

**Baseline Models (5):**
- ResNet50 (CNN)
- MobileNetV2 (CNN)
- EfficientNet-B0 (CNN)
- BERT-tiny (NLP/Transformer)
- ViT-tiny (Vision Transformer)

**Total: 14 models × 2 precisions = 28 ONNX files**

In [2]:
!{sys.executable} analysis/model_prep.py
print("\nModel preparation complete.")

FP32 model already exists: GCN_fp32.onnx
FP32 model already exists: GAT_fp32.onnx
  NNCF quantization failed: ONNX Quantize/Dequantize pairs only support input_high == output_high and input_low == output_low.
Skipping INT8 (previous failure): GAT_int8.onnx
FP32 model already exists: GATv2_fp32.onnx
  NNCF quantization failed: ONNX Quantize/Dequantize pairs only support input_high == output_high and input_low == output_low.
Skipping INT8 (previous failure): GATv2_int8.onnx
FP32 model already exists: GraphSAGE_fp32.onnx
FP32 model already exists: GIN_fp32.onnx
FP32 model already exists: SGC_fp32.onnx
FP32 model already exists: APPNP_fp32.onnx
FP32 model already exists: GraphTransformer_fp32.onnx
FP32 model already exists: MPNN_fp32.onnx
  NNCF quantization failed: ONNX Quantize/Dequantize pairs only support input_high == output_high and input_low == output_low.
Skipping INT8 (previous failure): GATv2_int8.onnx
  NNCF quantization failed: ONNX Quantize/Dequantize pairs only support input_

In [3]:
# Download required OGB datasets (ogbn-arxiv, ogbn-proteins, ogbn-products)
import sys
from pathlib import Path

data_dir = Path("data")
expected_dirs = ["ogbn_arxiv", "ogbn_proteins", "ogbn_products"]
missing = [d for d in expected_dirs if not (data_dir / d).exists()]

# Set True only if you really want to re-download/update datasets.
FORCE_DATASET_DOWNLOAD = False

if not missing:
    print("OGB datasets already present; skipping download/update.")
    print("Found:")
    for d in expected_dirs:
        print(f"  - {d}")
else:
    print("Missing datasets:")
    for d in missing:
        print(f"  - {d}")
    if FORCE_DATASET_DOWNLOAD:
        !{sys.executable} datasetPrep.py
    else:
        print("Skipping download. Set FORCE_DATASET_DOWNLOAD = True to fetch missing datasets.")

print("\nDataset preparation complete.")

OGB datasets already present; skipping download/update.
Found:
  - ogbn_arxiv
  - ogbn_proteins
  - ogbn_products

Dataset preparation complete.


## Phase 2: Per-Model Benchmarking

Each model cell runs the full analysis for that model and writes outputs under `results/<model>`.

What happens inside a model cell:
1. Multi-dataset sweep (ogbn-arxiv, ogbn-proteins, ogbn-products)
2. Devices: CPU, GPU, NPU (with safe skips for unstable cases)
3. HW comparison (CPU vs GPU vs NPU)
4. CPU fallback summary (from profiling)
5. Energy summary (from logs if available, otherwise estimated)

**Test Mode (`TEST_MODE = True` in Cell 2):**
- All benchmarks use **5 iterations, 1 repeat**
- Device + dataset coverage is reduced
- Use this to verify the pipeline works before running the full experiment.

In [ ]:
# Per-model benchmark helper: run all analyses in a single cell.
from analysis.scalability_analyzer import ScalabilityConfig, MultiModelPipeline
from analysis.hw_comparison import HWComparator
from analysis.energy_analyzer import EnergyAnalyzer
from pathlib import Path
import json
import numpy as np

RESULTS_ROOT = RESULTS_DIR.resolve()

def _resolve_model_paths(model_stem: str, allow_int8: bool = True):
    paths = []
    for suffix in ("_fp32.onnx", "_int8.onnx"):
        if suffix == "_int8.onnx" and not allow_int8:
            continue
        p = MODELS_DIR / f"{model_stem}{suffix}"
        if p.exists():
            paths.append(p.resolve())
    return paths

def _model_root(model_stem: str) -> Path:
    root = RESULTS_ROOT / model_stem
    root.mkdir(parents=True, exist_ok=True)
    return root

def _read_metadata(model_dir: Path) -> dict:
    for p in sorted(model_dir.glob("run_*/input_metadata.json")):
        try:
            payload = json.loads(p.read_text())
            return {
                k: float(payload.get(k))
                for k in ("used_num_nodes", "used_num_edges")
                if isinstance(payload.get(k), (int, float))
            }
        except Exception:
            continue
    return {}

def _summarize_model_results(model_root: Path) -> pd.DataFrame:
    matrices = list(model_root.rglob("scalability_matrix.csv"))
    if not matrices:
        return pd.DataFrame()

    rows = []
    for matrix in matrices:
        try:
            df = pd.read_csv(matrix)
        except Exception:
            continue

        parts = matrix.parts
        device = next((p for p in parts if p.startswith("device_")), "")
        dataset = next((p for p in parts if p.startswith("dataset_")), "")
        if device:
            df["device"] = device.replace("device_", "")
        if dataset:
            df["dataset"] = dataset.replace("dataset_", "")

        if "model" in df.columns:
            meta_rows = []
            for m in df["model"].astype(str):
                meta_rows.append(_read_metadata(matrix.parent / m))
            if meta_rows:
                df = pd.concat([df, pd.DataFrame(meta_rows)], axis=1)

        if "used_num_nodes" in df.columns and "used_num_edges" in df.columns:
            df["edges_per_node"] = df["used_num_edges"] / df["used_num_nodes"].replace(0, np.nan)

        df["base_model"] = model_root.name
        rows.append(df)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)

def _write_model_summary(model_root: Path, summary_df: pd.DataFrame) -> None:
    if summary_df.empty:
        return

    summary_path = model_root / "summary.csv"
    summary_df.to_csv(summary_path, index=False)

    global_path = RESULTS_ROOT / "benchmark_summary.csv"
    if global_path.exists():
        gdf = pd.read_csv(global_path)
        gdf = pd.concat([gdf, summary_df], ignore_index=True)
    else:
        gdf = summary_df.copy()

    key_cols = [c for c in ("base_model", "model", "device", "dataset") if c in gdf.columns]
    if key_cols:
        gdf = gdf.drop_duplicates(subset=key_cols, keep="last")
    gdf.to_csv(global_path, index=False)

def _write_fallback_summary(model_root: Path, summary_df: pd.DataFrame) -> None:
    if summary_df.empty or "o_cpu_fallback_pct" not in summary_df.columns:
        return

    fallback_df = summary_df.groupby("model", as_index=False)["o_cpu_fallback_pct"].mean()
    fallback_df = fallback_df.rename(columns={"o_cpu_fallback_pct": "cpu_fallback_pct"})
    fallback_df.to_csv(model_root / "cpu_fallback.csv", index=False)

    global_path = RESULTS_ROOT / "cpu_fallback.csv"
    if global_path.exists():
        gdf = pd.read_csv(global_path)
        gdf = pd.concat([gdf, fallback_df], ignore_index=True)
    else:
        gdf = fallback_df.copy()

    gdf = gdf.groupby("model", as_index=False)["cpu_fallback_pct"].mean()
    gdf.to_csv(global_path, index=False)

def _write_energy_summary(model_root: Path, summary_df: pd.DataFrame) -> None:
    if not RUN_ENERGY_ANALYSIS or summary_df.empty or "model" not in summary_df.columns:
        return

    rows = []
    for model_name in sorted(summary_df["model"].dropna().astype(str).unique()):
        csv_candidates = sorted(RESULTS_ROOT.glob(f"energy_log_hw_comp_{model_name}*.csv"))
        res = None
        if csv_candidates:
            try:
                res = EnergyAnalyzer(Path(csv_candidates[0])).analyze()
            except Exception:
                res = None

        if not res or "error" in res:
            model_rows = summary_df[summary_df["model"].astype(str) == model_name]
            if "device" in model_rows.columns:
                npu_rows = model_rows[model_rows["device"].astype(str).str.upper() == "NPU"]
            else:
                npu_rows = model_rows

            if not npu_rows.empty and "o_mean_ms" in npu_rows.columns:
                o_mean = float(npu_rows["o_mean_ms"].mean())
            elif "o_mean_ms" in model_rows.columns:
                o_mean = float(model_rows["o_mean_ms"].mean())
            else:
                o_mean = 0.0

            res = EnergyAnalyzer().estimate(o_mean)

        res["model"] = model_name
        rows.append(res)

    if rows:
        energy_df = pd.DataFrame(rows)
        energy_df.to_csv(model_root / "energy.csv", index=False)

def _run_hw_comparison(model_paths: list[Path], model_root: Path, iters: int, reps: int) -> None:
    if not RUN_HW_COMPARISON:
        return

    for model_path in model_paths:
        try:
            comparator = HWComparator(
                model_path,
                model_root,
                iterations=iters,
                repeats=reps,
                input_source="ogbn-arxiv",
                dataset_root=DATA_DIR.resolve(),
                flat_output=True,
                energy_log_dir=RESULTS_ROOT,
            )
            comparator.run()
        except Exception as exc:
            print(f"  [warn] HW comparison failed for {model_path.name}: {exc}")

def run_single_model_sweep(model_stem: str):
    model_paths_all = _resolve_model_paths(model_stem)
    if not model_paths_all:
        print(f"[skip] No ONNX files for {model_stem}")
        return

    model_root = _model_root(model_stem)

    if TEST_MODE:
        iters = 5
        reps = 1
        prof = False
        test_datasets = ["ogbn-arxiv"]
        test_devices = ["CPU"]
        print("\n[TEST MODE] Per-model run: 1 dataset, 1 device, 5 iters, 1 repeat")
    else:
        iters = 100
        reps = 3
        prof = True
        test_datasets = ["ogbn-arxiv", "ogbn-proteins", "ogbn-products"]
        test_devices = ["CPU", "GPU", "NPU"]
        print("\n[FULL MODE] Per-model run: 3 datasets, 3 devices, 100 iters, 3 repeats")

    if SKIP_UNSTABLE_NPU_MODELS and model_stem in UNSTABLE_NPU_MODELS:
        if "NPU" in test_devices:
            test_devices = [d for d in test_devices if d != "NPU"]
            print(f"  [safe] Skipping NPU for {model_stem} (known driver crash).")

    print(f"\n{'='*60}")
    print(f"Starting model run: {model_stem}")
    print(f"  iterations={iters}, repeats={reps}, profile={prof}")
    print(f"  devices={test_devices} | datasets={test_datasets}")
    print(f"  models={len(model_paths_all)} file(s)")
    print(f"  output={model_root}")
    print(f"{'='*60}\n")

    for device in test_devices:
        allow_int8 = True
        if device == "GPU" and SKIP_GPU_INT8:
            allow_int8 = False
        if device == "NPU" and SKIP_NPU_INT8:
            allow_int8 = False
        model_paths = _resolve_model_paths(model_stem, allow_int8=allow_int8)
        if not model_paths:
            print(f"[skip] No compatible ONNX files for {model_stem} on {device}")
            continue

        dev_root = model_root / f"device_{device}"
        dev_root.mkdir(parents=True, exist_ok=True)
        for ds in test_datasets:
            out_dir = dev_root / f"dataset_{ds}"
            out_dir.mkdir(parents=True, exist_ok=True)
            print(f"=== model={model_stem} | device={device} | dataset={ds} ===")
            scfg = ScalabilityConfig(
                models=model_paths,
                results_dir=out_dir,
                device=device,
                iterations=iters,
                warmup_iterations=5,
                repeats=reps,
                enable_profiling=prof,
                input_source=ds,
                dataset_root=DATA_DIR.resolve(),
            )
            MultiModelPipeline(scfg).run()

    _run_hw_comparison(model_paths_all, model_root, iters, reps)

    summary_df = _summarize_model_results(model_root)
    if summary_df.empty:
        print("[warn] No results found to summarize.")
    else:
        _write_model_summary(model_root, summary_df)
        _write_fallback_summary(model_root, summary_df)
        _write_energy_summary(model_root, summary_df)

    print(f"\n[OK] Model run complete: {model_stem}")

In [5]:
# Model 1: APPNP
run_single_model_sweep("APPNP")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: APPNP
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=APPNP | device=CPU | dataset=ogbn-arxiv ===
Skipping APPNP_fp32.onnx (already done)
Skipping APPNP_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=APPNP | device=CPU | dataset=ogbn-proteins ===
Skipping APPNP_fp32.onnx (already done)
Skipping APPNP_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_o

In [6]:
# Model 2: GAT
run_single_model_sweep("GAT")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: GAT
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=1 file(s)

=== model=GAT | device=CPU | dataset=ogbn-arxiv ===
Skipping GAT_fp32.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=GAT | device=CPU | dataset=ogbn-proteins ===
Skipping GAT_fp32.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-proteins\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Bas

In [ ]:
# Model 3: GATv2
run_single_model_sweep("GATv2")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: GATv2
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=1 file(s)

=== model=GATv2 | device=CPU | dataset=ogbn-arxiv ===
Skipping GATv2_fp32.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=GATv2 | device=CPU | dataset=ogbn-proteins ===
Skipping GATv2_fp32.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-proteins\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_

: 

In [ ]:
# Model 4: GCN
run_single_model_sweep("GCN")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: GCN
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=GCN | device=CPU | dataset=ogbn-arxiv ===
Skipping GCN_fp32.onnx (already done)
Skipping GCN_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=GCN | device=CPU | dataset=ogbn-proteins ===
Skipping GCN_fp32.onnx (already done)
Skipping GCN_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-proteins\s

In [4]:
# Model 5: GIN
run_single_model_sweep("GIN")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: GIN
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=GIN | device=CPU | dataset=ogbn-arxiv ===
Skipping GIN_fp32.onnx (already done)
Skipping GIN_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=GIN | device=CPU | dataset=ogbn-proteins ===
Skipping GIN_fp32.onnx (already done)
Skipping GIN_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-proteins\s

In [5]:
# Model 6: GraphSAGE
run_single_model_sweep("GraphSAGE")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: GraphSAGE
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=GraphSAGE | device=CPU | dataset=ogbn-arxiv ===
Skipping GraphSAGE_fp32.onnx (already done)
Skipping GraphSAGE_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=GraphSAGE | device=CPU | dataset=ogbn-proteins ===
Skipping GraphSAGE_fp32.onnx (already done)
Skipping GraphSAGE_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\densit

In [ ]:
# Model 7: GraphTransformer
run_single_model_sweep("GraphTransformer")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: GraphTransformer
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=GraphTransformer | device=CPU | dataset=ogbn-arxiv ===
Skipping GraphTransformer_fp32.onnx (already done)
Skipping GraphTransformer_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=GraphTransformer | device=CPU | dataset=ogbn-proteins ===
Skipping GraphTransformer_fp32.onnx (already done)
Skipping GraphTransformer_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\P

: 

In [ ]:
# Model 8: MPNN
run_single_model_sweep("MPNN")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: MPNN
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=MPNN | device=CPU | dataset=ogbn-arxiv ===
Skipping MPNN_fp32.onnx (already done)
Skipping MPNN_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=MPNN | device=CPU | dataset=ogbn-proteins ===
Skipping MPNN_fp32.onnx (already done)
Skipping MPNN_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-pro

In [3]:
# Model 9: SGC
run_single_model_sweep("SGC")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: SGC
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=SGC | device=CPU | dataset=ogbn-arxiv ===
Skipping SGC_fp32.onnx (already done)
Skipping SGC_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=SGC | device=CPU | dataset=ogbn-proteins ===
Skipping SGC_fp32.onnx (already done)
Skipping SGC_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-proteins\s

In [4]:
# Model 10: resnet50
run_single_model_sweep("resnet50")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: resnet50
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=resnet50 | device=CPU | dataset=ogbn-arxiv ===
Skipping resnet50_fp32.onnx (already done)
Skipping resnet50_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=resnet50 | device=CPU | dataset=ogbn-proteins ===
Skipping resnet50_fp32.onnx (already done)
Skipping resnet50_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep

In [5]:
# Model 11: mobilenetv2
run_single_model_sweep("mobilenetv2")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: mobilenetv2
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=mobilenetv2 | device=CPU | dataset=ogbn-arxiv ===
Skipping mobilenetv2_fp32.onnx (already done)
Skipping mobilenetv2_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=mobilenetv2 | device=CPU | dataset=ogbn-proteins ===
Skipping mobilenetv2_fp32.onnx (already done)
Skipping mobilenetv2_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\

In [6]:
# Model 12: efficientnet-b0
run_single_model_sweep("efficientnet-b0")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: efficientnet-b0
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=efficientnet-b0 | device=CPU | dataset=ogbn-arxiv ===
Skipping efficientnet-b0_fp32.onnx (already done)
Skipping efficientnet-b0_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=efficientnet-b0 | device=CPU | dataset=ogbn-proteins ===
Skipping efficientnet-b0_fp32.onnx (already done)
Skipping efficientnet-b0_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects

In [7]:
# Model 13: bert-tiny
run_single_model_sweep("bert-tiny")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: bert-tiny
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=bert-tiny | device=CPU | dataset=ogbn-arxiv ===
Skipping bert-tiny_fp32.onnx (already done)
Skipping bert-tiny_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=bert-tiny | device=CPU | dataset=ogbn-proteins ===
Skipping bert-tiny_fp32.onnx (already done)
Skipping bert-tiny_int8.onnx (already done)

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\densit

In [8]:
# Model 14: vit-tiny
run_single_model_sweep("vit-tiny")


[FULL MODE] Density sweep: 3 datasets, 3 devices, 100 iters, 3 repeats

Starting model sweep: vit-tiny
  iterations=100, repeats=3, profile=True
  devices=['CPU', 'GPU', 'NPU'] | datasets=['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
  models=2 file(s)

=== model=vit-tiny | device=CPU | dataset=ogbn-arxiv ===
Skipping vit-tiny_fp32.onnx (already done)
Processing: vit-tiny_int8.onnx
[2026-05-12 23:28:02] start mode=baseline model=vit-tiny_int8.onnx device=CPU warmup=5 iters=100
[2026-05-12 23:28:02] creating session
  ❌ Error benchmarking vit-tiny_int8.onnx: [ONNXRuntimeError] : 1 : FAIL : This is an invalid model. Error: the graph is not acyclic.

[OK] Scalability Analysis Complete. Results saved to C:\Users\yusuf\Projects\npu-graph-opt-benchmarking\results\density_sweep\device_CPU\dataset_ogbn-arxiv\scalability_matrix.csv
  -> Exported profiling trace for [vit-tiny_fp32] (Baseline) to results/
  -> Exported profiling trace for [vit-tiny_fp32] (Optimized) to results/
=== model=vit-

## Phase 3: Optional Aggregates

Per-model cells already produce summaries under `results/<model>` and a global `results/benchmark_summary.csv`.

Use the cells below only if you want to refresh the aggregate CSVs used by the figures. No extra
analysis folders are created.

In [ ]:
# Analysis 1: Density Aggregation (from per-model summaries)
from pathlib import Path
import numpy as np
import pandas as pd

summary_path = Path("results/benchmark_summary.csv")
out_path = Path("results/density_analysis.csv")

if not summary_path.exists():
    print(f"ERROR: {summary_path} not found. Run Phase 2 model cells first.")
    density_df = pd.DataFrame(columns=["dataset", "model", "edges_per_node", "o_mean_ms", "device"])
    density_df.to_csv(out_path, index=False)
    print(f"Empty density data saved to {out_path}")
else:
    df = pd.read_csv(summary_path)
    if "edges_per_node" not in df.columns:
        if "used_num_nodes" in df.columns and "used_num_edges" in df.columns:
            df["edges_per_node"] = df["used_num_edges"] / df["used_num_nodes"].replace(0, np.nan)

    keep_cols = [c for c in ["dataset", "model", "edges_per_node", "o_mean_ms", "device"] if c in df.columns]
    density_df = df[keep_cols].copy()
    density_df.to_csv(out_path, index=False)

    if not density_df.empty and "dataset" in density_df.columns:
        summary = density_df.groupby("dataset").agg(
            edges_per_node=("edges_per_node", "mean"),
            latency_ms=("o_mean_ms", "mean"),
        ).sort_values("edges_per_node")
        print(f"\nDensity data: {len(density_df)} total records")
        print(summary)

Found 0 dataset result directories

No density data found - run Phase 2 first
Creating empty density data...
Empty density data saved to results\figures\density_analysis.csv


In [ ]:
# Analysis 2: Graph Topology Statistics
from analysis.graph_topology_analyzer import GraphTopologyAnalyzer

# Ensure out_dir exists
if 'out_dir' not in locals():
    out_dir = Path('results/figures')
    out_dir.mkdir(parents=True, exist_ok=True)

# Check for OGB datasets
try:
    analyzer = GraphTopologyAnalyzer(results_dir=out_dir, dataset_root=Path('data'))
    
    # Try each dataset individually to skip any corrupted ones
    datasets = ['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products']
    stats_list = []
    for ds in datasets:
        try:
            print(f"Loading {ds}...")
            stats = analyzer.compute_statistics(ds)
            stats_list.append(stats.to_dict())
            print(f"  OK: {ds} loaded")
        except Exception as ds_e:
            print(f"  Skipping {ds}: {ds_e}")
    
    if stats_list:
        stats_df = pd.DataFrame(stats_list)
        # Generate Figure 2 if not already present
        fig2_path = out_dir / 'fig2_degree_distribution_loglog.png'
        if not fig2_path.exists():
            try:
                analyzer.analyze_degree_distribution([s['dataset_name'] for s in stats_list])
            except Exception as plot_e:
                print(f"  Warning: Could not generate degree plot: {plot_e}")
    else:
        stats_df = None
        print("No statistics generated - all datasets failed to load")
    
    if stats_df is not None and not stats_df.empty:
        print("\nDataset Statistics:")
        cols = ['dataset_name', 'num_nodes', 'num_edges', 'avg_degree', 'density']
        if 'power_law_alpha' in stats_df.columns:
            cols.append('power_law_alpha')
        print(stats_df[cols].to_string(index=False))
    else:
        print("No statistics generated")
except Exception as e:
    print(f"Error in topology analysis: {e}")
    print("This is normal if datasets haven't been downloaded yet.")
    stats_df = None

Loading ogbn-arxiv...
Loading dataset: ogbn-arxiv...
  Loaded in 97.47s - 169,343 nodes, 1,166,243 edges
  OK: ogbn-arxiv loaded
Loading ogbn-proteins...
  Skipping ogbn-proteins: Unknown dataset: ogbn-proteins
Loading ogbn-products...
Loading dataset: ogbn-products...


In [ ]:
# Analysis 3: Operator Composition
import onnx
from analysis.plot_config import get_model_category

# Define categories for ONNX operators
categories = ['SpMM/MatMul', 'MLP', 'Activation', 'Attention', 'Memory/Shape', 'Other']

def categorize(op_type):
    op = str(op_type)
    if op in {'MatMul', 'Gemm'}:
        return 'SpMM/MatMul'
    if op in {'Conv', 'ConvTranspose'}:
        return 'MLP'
    if 'Attention' in op or op in {'Softmax', 'LayerNormalization'}:
        return 'Attention'
    if op in {'Relu', 'Sigmoid', 'Tanh', 'Gelu'}:
        return 'Activation'
    if op in {'Gather', 'Scatter', 'Slice', 'Concat', 'Transpose', 'Reshape'}:
        return 'Memory/Shape'
    return 'Other'

# Find all ONNX models
model_files = sorted(Path('models').glob('*_fp32.onnx'))
print(f"Analyzing {len(model_files)} ONNX models...")

rows = []
for model_path in model_files:
    try:
        model = onnx.load(str(model_path))
        counts = {c: 0 for c in categories}
        for node in model.graph.node:
            counts[categorize(node.op_type)] += 1
        total = sum(counts.values()) or 1
        row = {k: (v/total)*100 for k, v in counts.items()}
        row['model'] = model_path.stem
        row['category'] = get_model_category(model_path.stem)
        rows.append(row)
    except Exception as e:
        print(f"  Error analyzing {model_path.name}: {e}")

if rows:
    op_df = pd.DataFrame(rows).sort_values(['category', 'model'])
    out_dir = Path('results/figures')
    op_df.to_csv(out_dir / 'operator_analysis.csv', index=False)
    print(f"\nOperator analysis: {len(op_df)} models")
    display_cols = ['model', 'category', 'SpMM/MatMul', 'MLP', 'Attention']
    print(op_df[display_cols].head(10).to_string(index=False))
else:
    print("\nNo operator data - check if models exist in ./models/")
    print("Creating empty operator data...")
    # Create empty operator data so downstream cells can handle it
    op_df = pd.DataFrame(columns=['model', 'category'] + categories)
    out_dir = Path('results/figures')
    out_dir.mkdir(parents=True, exist_ok=True)
    op_df.to_csv(out_dir / 'operator_analysis.csv', index=False)
    print(f"Empty operator data saved to {out_dir / 'operator_analysis.csv'}")

## Phase 4: Visualizations

Six publication-ready figures. Each cell generates PNG (300 DPI) + SVG outputs.

In [ ]:
# Figure 1: Density vs Performance
# Shows relationship between graph density and NPU latency

import numpy as np
from analysis.plot_config import apply_ieee_style, savefig_ieee, IEEE_COLORS, SINGLE_COL

apply_ieee_style()
fig, ax = plt.subplots(figsize=SINGLE_COL)

# Try to load density data if not in memory
def load_density_data():
    """Load density data from memory or CSV file."""
    global density_df
    
    # Check if already in memory
    if 'density_df' in locals() and density_df is not None:
        return density_df
    
    # Try to load from CSV
    csv_path = Path('results/figures/density_analysis.csv')
    if csv_path.exists():
        try:
            density_df = pd.read_csv(csv_path)
            print(f"Loaded density data from {csv_path}")
            return density_df
        except Exception as e:
            print(f"Error loading CSV: {e}")
            return None
    
    return None

# Load the data
density_df = load_density_data()

# Check if data exists
if density_df is None:
    print("ERROR: No density data. Run Analysis 1 (Cell 9) first.")
    print("Or: Run Phase 2 (Benchmarking) to generate results.")
    # Create placeholder figure
    ax.text(0.5, 0.5, 'Density Data Not Available\n\nRun Phase 2 (Benchmarking) to generate data.', 
            ha='center', va='center', fontsize=10, style='italic', color='gray',
            transform=ax.transAxes)
    ax.set_title('Figure 1: Density vs Performance (No Data)', fontsize=10)
    ax.axis('off')
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig1_density_vs_performance')
    plt.close(fig)
    print("Placeholder figure saved.")
elif density_df.empty:
    print("INFO: density_df is empty")
    ax.text(0.5, 0.5, 'Density Data Empty\n\nNo density data available.', 
            ha='center', va='center', fontsize=10, style='italic', color='gray',
            transform=ax.transAxes)
    ax.set_title('Figure 1: Density vs Performance (Empty)', fontsize=10)
    ax.axis('off')
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig1_density_vs_performance')
    plt.close(fig)
    print("Placeholder figure saved.")
else:
    # Filter NPU results
    npu_data = density_df[density_df.get('device', 'NPU') == 'NPU']
    if npu_data.empty:
        print("WARNING: No NPU data found, showing all data")
        npu_data = density_df
    
    # Plot each dataset
    for i, (dataset, group) in enumerate(npu_data.groupby('dataset')):
        if group.get('edges_per_node').notna().any():
            x = group['edges_per_node'].mean()
            y = group['o_mean_ms'].mean()
            ax.scatter(x, y, s=80, color=IEEE_COLORS[i % len(IEEE_COLORS)], 
                      label=dataset, edgecolors='black', linewidth=0.5)
    
    ax.set_xlabel('Average Degree (edges/node)', fontsize=9)
    ax.set_ylabel('Latency (ms)', fontsize=9)
    ax.set_title('Figure 1: Density vs Performance', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig1_density_vs_performance')
    display(Image(filename=out_dir / 'fig1_density_vs_performance.png'))

plt.close(fig)

In [ ]:
# Figure 2: Degree Distribution (Log-Log)
# Shows power-law behavior of graph degree distributions

out_dir = Path('results/figures')
fig_path = out_dir / 'fig2_degree_distribution_loglog.png'

if fig_path.exists():
    display(Image(filename=fig_path))
    print(f"\nFigure loaded from: {fig_path}")
else:
    print("Degree distribution figure not found.")
    print("Run Analysis 2 (Cell 11) to generate, which requires OGB datasets.")
    print("Note: Datasets auto-download on first run (requires internet).")
    print("If a dataset is corrupted, delete its folder under data/ and re-run.")

In [ ]:
# Figure 3: Operator Breakdown (Stacked Bars)
# Compares operator composition between GNN, CNN, and Transformer models

from analysis.plot_config import DOUBLE_COL, shorten_label

# Try to load operator data if not in memory
def load_operator_data():
    """Load operator data from memory or CSV file."""
    global op_df
    
    # Check if already in memory
    if 'op_df' in locals() and op_df is not None:
        return op_df
    
    # Try to load from CSV
    csv_path = Path('results/figures/operator_analysis.csv')
    if csv_path.exists():
        try:
            op_df = pd.read_csv(csv_path)
            print(f"Loaded operator data from {csv_path}")
            return op_df
        except Exception as e:
            print(f"Error loading CSV: {e}")
            return None
    
    return None

# Load the data
op_df = load_operator_data()

if op_df is None:
    print("ERROR: No operator data. Run Analysis 3 (Cell 11) first.")
    print("Or: Check if ONNX models exist in ./models/")
    # Create placeholder figure
    fig, ax = plt.subplots(figsize=DOUBLE_COL)
    ax.text(0.5, 0.5, 'Operator Data Not Available\n\nRun Analysis 3 to generate data.\nEnsure ONNX models exist in ./models/', 
            ha='center', va='center', fontsize=10, style='italic', color='gray',
            transform=ax.transAxes)
    ax.set_title('Figure 3: Operator Breakdown (No Data)', fontsize=10)
    ax.axis('off')
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig3_operator_breakdown')
    plt.close(fig)
    print("Placeholder figure saved.")
elif op_df.empty:
    print("INFO: op_df is empty")
    fig, ax = plt.subplots(figsize=DOUBLE_COL)
    ax.text(0.5, 0.5, 'Operator Data Empty\n\nNo operator data available.', 
            ha='center', va='center', fontsize=10, style='italic', color='gray',
            transform=ax.transAxes)
    ax.set_title('Figure 3: Operator Breakdown (Empty)', fontsize=10)
    ax.axis('off')
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig3_operator_breakdown')
    plt.close(fig)
    print("Placeholder figure saved.")
else:
    fig, ax = plt.subplots(figsize=DOUBLE_COL)
    x = np.arange(len(op_df))
    bottom = np.zeros(len(op_df))
    colors = {cat: IEEE_COLORS[i % len(IEEE_COLORS)] for i, cat in enumerate(categories)}
    
    for cat in categories:
        vals = op_df[cat].fillna(0).values
        ax.bar(x, vals, bottom=bottom, label=cat, color=colors[cat], edgecolor='k', linewidth=0.25)
        bottom += vals
    
    # Category separators
    last_cat = None
    for i, cat in enumerate(op_df['category']):
        if cat != last_cat and i > 0:
            ax.axvline(i - 0.5, color='gray', lw=0.5, ls='--')
        last_cat = cat
    
    ax.set_xticks(x)
    ax.set_xticklabels([shorten_label(s, 12) for s in op_df['model']], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Share of ONNX nodes (%)', fontsize=9)
    ax.set_title('Figure 3: Operator Breakdown', fontsize=10)
    ax.legend(ncol=3, fontsize=7, loc='upper right')
    ax.set_ylim(0, 120)
    
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig3_operator_breakdown')
    display(Image(filename=out_dir / 'fig3_operator_breakdown.png'))
    plt.close(fig)

In [ ]:
# Figure 4: CPU Fallback Heatmap
# Shows which models/operators fall back to CPU execution

from analysis.plot_config import DOUBLE_COL

# Try to load fallback data if not in memory
def load_fallback_data():
    """Load CPU fallback data from memory or CSV file."""
    global fallback_df
    
    # Check if already in memory
    if 'fallback_df' in locals() and fallback_df is not None:
        return fallback_df
    
    # Try to load from CSV
    csv_path = Path('results/figures/cpu_fallback_analysis.csv')
    if csv_path.exists():
        try:
            fallback_df = pd.read_csv(csv_path)
            print(f"Loaded CPU fallback data from {csv_path}")
            return fallback_df
        except Exception as e:
            print(f"Error loading CSV: {e}")
            return None
    
    return None

# Load the data
fallback_df = load_fallback_data()

if fallback_df is None:
    print("ERROR: No CPU fallback data. Run Analysis 4 (Cell 12) first.")
    print("Or: Run benchmarks with --profile flag")
elif fallback_df.empty:
    print("INFO: No CPU fallback profiling data available.")
    print("This is expected in TEST_MODE (profiling disabled).")
    print("Run with TEST_MODE = False and enable_profiling=True to generate fallback data.")
    
    # Create a placeholder figure with info message
    fig, ax = plt.subplots(figsize=DOUBLE_COL)
    ax.text(0.5, 0.5, 'CPU Fallback Data Not Available\n\nProfiling disabled in TEST_MODE.\nRun full benchmarks with --profile to generate this data.', 
            ha='center', va='center', fontsize=10, style='italic', color='gray',
            transform=ax.transAxes)
    ax.set_title('Figure 4: CPU Fallback by Model (No Data)', fontsize=10)
    ax.axis('off')
    
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig4_cpu_fallback')
    plt.close(fig)
    print(f"Placeholder figure saved.")
else:
    fig, ax = plt.subplots(figsize=DOUBLE_COL)
    
    models = fallback_df['model'].tolist()
    values = fallback_df['cpu_fallback_pct'].tolist()
    
    colors = ['green' if v < 10 else 'orange' if v < 50 else 'red' for v in values]
    ax.barh(models, values, color=colors, edgecolor='black', linewidth=0.3)
    
    ax.set_xlabel('CPU Fallback (%)', fontsize=9)
    ax.set_ylabel('Model', fontsize=9)
    ax.set_title('Figure 4: CPU Fallback by Model', fontsize=10)
    ax.axvline(10, color='green', linestyle='--', linewidth=0.8, alpha=0.5, label='Low (<10%)')
    ax.axvline(50, color='orange', linestyle='--', linewidth=0.8, alpha=0.5, label='Medium (<50%)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis='x')
    
    out_dir = Path('results/figures')
    savefig_ieee(fig, out_dir / 'fig4_cpu_fallback')
    display(Image(filename=out_dir / 'fig4_cpu_fallback.png'))
    plt.close(fig)

In [ ]:
# Figure 5: Fusion Gain vs Performance
# Correlation between operator fusion and latency improvement

from scipy import stats
from analysis.plot_config import SINGLE_COL

# Find the density-sweep scalability matrix (works for both test and full mode)
matrix_candidates = list(Path('results/density_sweep').rglob('scalability_matrix.csv'))
if matrix_candidates:
    matrix_csv = matrix_candidates[0]
else:
    matrix_csv = Path('results/scalability_matrix.csv')

if not matrix_csv.exists():
    print("ERROR: Scalability matrix not found. Run Phase 2 (Benchmarking) first.")
    print(f"Expected: {matrix_csv}")
else:
    try:
        df = pd.read_csv(matrix_csv)
        df['lat_impr_pct'] = (df['b_mean_ms'] - df['o_mean_ms']) / df['b_mean_ms'].replace(0, np.nan) * 100
        df['category'] = df['model'].apply(get_model_category)
        
        fig, ax = plt.subplots(figsize=SINGLE_COL)
        markers = {'GNN (Irregular)': 'o', 'CNN (Regular)': 's', 'Transformer (Global Attn)': '^'}
        
        for cat in df['category'].unique():
            cat_data = df[df['category'] == cat]
            ax.scatter(cat_data['speedup'], cat_data['lat_impr_pct'],
                      s=50, alpha=0.8, label=cat, marker=markers.get(cat, 'o'),
                      edgecolors='black', linewidth=0.5)
        
        ax.axvline(1.0, color='black', linestyle='--', linewidth=0.8)
        ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.6)
        
        # Correlation
        valid = df[df['speedup'].notna() & df['lat_impr_pct'].notna()]
        if len(valid) > 3:
            r, p = stats.pearsonr(valid['speedup'], valid['lat_impr_pct'])
            ax.text(0.05, 0.95, f'r={r:.2f}, p={p:.3f}', transform=ax.transAxes,
                   fontsize=8, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
        
        ax.set_xlabel('Fusion Gain Ratio', fontsize=9)
        ax.set_ylabel('Latency Improvement (%)', fontsize=9)
        ax.set_title('Figure 5: Fusion Gain vs Performance', fontsize=10)
        ax.legend(fontsize=6)
        ax.grid(True, alpha=0.3)
        
        out_dir = Path('results/figures')
        savefig_ieee(fig, out_dir / 'fig5_fusion_gain')
        display(Image(filename=out_dir / 'fig5_fusion_gain.png'))
        plt.close(fig)
    except Exception as e:
        print(f"ERROR plotting Figure 5: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# Figure 6: Scaling Analysis
# Shows O(N) vs O(E) scaling behavior for GNN workloads

from analysis.plot_config import DOUBLE_COL

scaling_csv = Path('results/scaling_sweep/scaling_sweep.csv')

if not scaling_csv.exists():
    print("ERROR: Scaling data not found. Run Phase 2 - Scaling Analysis (Cell 7) first.")
    print(f"Expected: {scaling_csv}")
else:
    try:
        df = pd.read_csv(scaling_csv)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=DOUBLE_COL)
        
        ax1.plot(df['num_nodes'], df['o_mean_ms'], 'o-', color=IEEE_COLORS[0], 
                linewidth=1.5, markersize=6, label='Measured')
        ax1.set_xlabel('Number of Nodes', fontsize=9)
        ax1.set_ylabel('Latency (ms)', fontsize=9)
        ax1.set_title('Scaling by Nodes', fontsize=9)
        ax1.grid(True, alpha=0.3)
        
        ax2.plot(df['num_edges'], df['o_mean_ms'], 's-', color=IEEE_COLORS[1],
                linewidth=1.5, markersize=6, label='Measured')
        ax2.set_xlabel('Number of Edges', fontsize=9)
        ax2.set_ylabel('Latency (ms)', fontsize=9)
        ax2.set_title('Scaling by Edges', fontsize=9)
        ax2.grid(True, alpha=0.3)
        
        fig.suptitle('Figure 6: Scaling Characteristics', fontsize=10)
        plt.tight_layout()
        
        out_dir = Path('results/figures')
        savefig_ieee(fig, out_dir / 'fig6_scaling')
        display(Image(filename=out_dir / 'fig6_scaling.png'))
        plt.close(fig)
        
        print(f"\nScaling summary:")
        print(df[['num_nodes', 'num_edges', 'o_mean_ms']].to_string(index=False))
    except Exception as e:
        print(f"ERROR plotting Figure 6: {e}")
        import traceback
        traceback.print_exc()

## Summary

Generated outputs in `results/figures/`:
- PNG (300 DPI) for publications
- SVG for vector editing
- CSV for data tables

In [ ]:
# Final Summary

print("=" * 60)
print("BENCHMARKING COMPLETE")
print("=" * 60)

out_dir = Path('results/figures')
if out_dir.exists():
    print(f"\nGenerated outputs in {out_dir}:")
    
    for ext in ['png', 'svg', 'csv']:
        files = sorted(out_dir.glob(f'*.{ext}'))
        if files:
            print(f"\n{ext.upper()} files ({len(files)} total):")
            for f in files[:10]:
                print(f"  - {f.name}")
            if len(files) > 10:
                print(f"  ... and {len(files)-10} more")

    print("\nAll figures exported as PNG (300 DPI) + SVG")
else:
    print("\nNo output directory found - run analysis cells first")